In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('hw6') \
    .getOrCreate()
spark.range(5).show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/09 21:21:17 WARN Utils: Your hostname, PradyunPC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/09 21:21:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 21:21:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/09 21:21:18 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/09 21:21:18 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [3]:
spark.version

'4.1.1'

In [4]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-09 21:30:53--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.170.186.41, 3.170.186.111, 3.170.186.198, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.170.186.41|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  78.5MB/s    in 0.9s    

2026-03-09 21:30:54 (78.5 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [37]:
df_yellow = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2025-11.parquet')

In [38]:
df_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [39]:
df_yellow = df_yellow.repartition(4)

In [40]:
df_yellow.write.parquet('data/hw6/', mode = "overwrite")

In [41]:
df_yellow.createOrReplaceTempView('trips_data')

In [42]:
spark.sql("""
SELECT COUNT(*)
FROM trips_data
WHERE DATE(tpep_pickup_datetime) = '2025-11-15'
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [43]:
spark.sql("""
SELECT MAX(
    (unix_timestamp(tpep_dropoff_datetime) -
     unix_timestamp(tpep_pickup_datetime)) / 3600
) AS longest_trip_hours
FROM trips_data
""").show()

[Stage 38:==========================================>             (13 + 4) / 17]

+------------------+
|longest_trip_hours|
+------------------+
| 90.64666666666666|
+------------------+



In [20]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-09 21:59:50--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.35.127.27, 13.35.127.9, 13.35.127.26, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.35.127.27|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-09 21:59:50 (358 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [44]:
df_zones = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

In [45]:
df_zones.printSchema()

root
 |-- LocationID: string (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [46]:
import pandas as pd

In [47]:
df_zones_pd = pd.read_csv('taxi_zone_lookup.csv')

In [48]:
df_zones_pd.dtypes

LocationID      int64
Borough           str
Zone              str
service_zone      str
dtype: object

In [49]:
spark.createDataFrame(df_zones_pd).schema

StructType([StructField('LocationID', LongType(), True), StructField('Borough', StringType(), True), StructField('Zone', StringType(), True), StructField('service_zone', StringType(), True)])

In [50]:
from pyspark.sql import types

In [51]:
zones_schema = types.StructType([
    types.StructField("LocationID", types.IntegerType(), True),
    types.StructField("Borough", types.StringType(), True),
    types.StructField("Zone", types.StringType(), True),
    types.StructField("service_zone", types.StringType(), True)
])

In [52]:
df_zones = spark.read \
    .option("header", "true") \
    .schema(zones_schema) \
    .csv('taxi_zone_lookup.csv')

In [56]:
df_zones \
        .repartition(4) \
        .write.parquet('data/hw6/pq/')

In [59]:
df_zones.createOrReplaceTempView('zones_data')

In [61]:
spark.sql("""
SELECT *
FROM zones_data
LIMIT 10
""").show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
+----------+-------------+--------------------+------------+



In [63]:
spark.sql("""
SELECT
    z.Zone,
    COUNT(*) AS trip_count
FROM trips_data y
JOIN zones_data z
ON y.PULocationID = z.LocationID
GROUP BY z.Zone
ORDER BY trip_count ASC
LIMIT 1
""").show()

[Stage 54:==============================================>         (14 + 3) / 17]

+--------------------+----------+
|                Zone|trip_count|
+--------------------+----------+
|Governor's Island...|         1|
+--------------------+----------+

